##### Parameters

In [ ]:
# Stores job metadata, source/target config, and execution 
# Initializes all runtime inputs and metadata required for the ETL task.
task_id = 50
task_name = 'Load to silver - Address'
source_settings = '{"schema_name": "sqldb_adventureworks", "table_name": "SalesLT__Address"}'
log_run_pairs = "[{'previous_task_executions_id': '1D993FD2-B0A7-4368-8640-AB136A515907', 'previous_lineage_id': 'E149B7C3-0410-479C-924D-72969FB58F15'}]"
target_settings = '{"schema_name": "sqldb_adventureworks", "table_name": "sales_lt__Address", "load_strategy": "overwrite", "primary_keys": ["AddressID"]}'
option_settings = None
limit_rows_for_debugging = False
akv_uri = None
tenant_id_secret_name = None
client_id_secret_name = None
client_secret_name = None
server = None
# metadata_db
metadata_database = None
database = None
parent_run_id = None

##### Spark Config
Ensures correct reading/writing of old date formats

Purpose: Prevents date corruption issues when working with older datasets.

In [ ]:
# Set Spark settings to enable reading of old dates
spark.conf.set("spark.sql.parquet.int96RebaseModeInRead","CORRECTED")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite","CORRECTED")

##### Configuration and imports

In [ ]:
import ast
import uuid
import json
from delta.tables import DeltaTable
import re
import pyspark.sql.functions as F # import when, col, lit, to_date, to_timestamp, trim
import datetime

##### Import shared functions and setup logging

1. Setup_logging() function: Sets up logging for the notebook so messages (info, warnings, errors) are logged in a consistent format.

2. Create DirectoryMaintenance class: Utility class to clean up empty folders in a data lake (Fabric/OneLake).

3. get_metadata_control_db_connection_info() function: Fetches metadata_database connection details securely from Azure Key Vault

4. Create SQLDatabase class: Provides a reusable way to connect to SQL Server and run queries using SQLAlchemy.


| Component                         | Type              | Purpose |
|----------------------------------|-------------------|--------|
| 1. setup_logging                    | Function          | Configure structured logging output for the notebook |
| 2. DirectoryMaintenance             | Class             | Manage and delete empty directories in storage |
| - delete_empty_directories         | Method            | Remove empty folders (deepest first) |
| 3. get_metadata_control_db_connection_info | Function | Retrieve secure database connection details from Key Vault |
| 4. get_fabric_sql_db_connection_info | Function | Retrieve secure database connection details from Key Vault |
| 5. SQLDatabase                      | Class             | Manage SQL Server connection and operations |
| - execute_query                    | Method            | Run SELECT queries and return results |
| - execute_non_query                | Method            | Run INSERT/UPDATE/DELETE queries |
| - execute_stored_procedure         | Method            | Execute stored procedures |
| - %run nb_shared_functions         | Notebook include  | Import shared reusable functions |
| - %run nb_json_utility_functions   | Notebook include  | Import JSON helper functions |
| - Logging setup (logger)           | Configuration     | Initialize logger for the pipeline |
| - list_files                       | Function          | Recursively list files with metadata, sorted by time |


In [ ]:
%run nb_shared_functions

In [ ]:
# From nb_shared_functions → prepares logging
setup_logging()
# getLogger() → from Python logging library → creates pipeline-specific logger
logger = logging.getLogger('LoadToSilver')
# Method of logger → controls what messages are shown
logger.setLevel(logging.INFO)


##### Data Transformation & Logging Functions

| Function | Type | Purpose | Input | Output |
|----------|------|--------|-------|--------|
| to_snake_case | Function | Convert column names to snake_case | string | string |
| trim_all_string_columns | Function | Trim all string columns | DataFrame | DataFrame |
| replace_old_dates | Function | Fix invalid dates | DataFrame | DataFrame |
| clean_bronze_table | Function | Apply all transformations | DataFrame | DataFrame |
| update_stage_status | Function | Update pipeline status in DB | status + metadata | None |


In [ ]:
# Define functions for data transformation
#1. Convert a string(column Names) to snake_case. Purpose: Standardize column naming
def to_snake_case(string):
    string = re.sub(r'([a-z])([A-Z0-9])', r'\1_\2', string)
    string = re.sub(r'([A-Z])([A-Z][a-z])', r'\1_\2', string)
    string = string.lower().strip()
    return string

#2. Remove leading/trailing spaces from all string column values. input: Spark DataFrame
def trim_all_string_columns(df):
    string_columns = [field.name for field in df.schema.fields if field.dataType.typeName() == 'string']
    df = df.select(
        *[F.trim(F.col(c)).alias(c) if c in string_columns else F.col(c) for c in df.columns]
    )
    return df

#3. Replace old dates: If date ≤ 1900-01-01 → replace with 1900-01-01. Input:DataFrame

def replace_old_dates(df):
    date_columns = [field.name for field in df.schema.fields if field.dataType.typeName() in('date','timestamp')]
    for date_column in date_columns:
        df = df.withColumn(
            date_column, 
		    F.when(
			    F.col(date_column) <= F.to_date(F.lit('1900-01-01')), F.to_date(F.lit('1900-01-01'), 'yyyy-MM-dd')
            )
			.otherwise(
                F.col(date_column)
            )
        )
    return df

#4.  Define parent function that applies all cleaning steps to the DataFrame.
def clean_bronze_table(df, limit_rows_for_debugging=False):
    if limit_rows_for_debugging:
        df = df.limit(1000)

    snake_case_column_names = [to_snake_case(col) for col in df.columns]
    df = df.toDF(*snake_case_column_names)

    #df = df.withColumn("ingested_date_time", (F.col("ingested_date_time").cast("timestamp")))

    df = replace_old_dates(df)

    df = trim_all_string_columns(df)

    return df



In [ ]:
# Moved to shared notebook : 5. Updates pipeline execution status in metadata_database.
def update_stage_status(status: str, log_run_data: list, metadata_db):
    """Update metadata_database records for current and previous stages"""
    
    #5.1: Status mappings
    status_config = {
        'Completed': {'current': 'Ready', 'previous': 'Processed'},
        'Failed': {'current': 'N/A', 'previous': 'Failed'},
        'No files to process': {'current': 'N/A', 'previous': 'No files to process'}
    }

    config = status_config.get(status, {'current': 'N/A', 'previous': 'N/A'})

    try:
        #5.2: Parse input
        records = ast.literal_eval(log_run_data) if isinstance(log_run_data, str) else log_run_data
    except Exception:
        logger.exception('log_run_data is not valid; skipping updates')
        records = []

    for record in records:
        # Update current stage
        try:
            logger.info(f"Updating current stage logs for {record['current_task_executions_id']}")
            #5.3: Call stored procedures
            metadata_db.execute_stored_procedure(
                #5.4:Current stage update
                'logging.usp_log_task_execution_completion',
                False,
                task_executions_id=record['current_task_executions_id'],
                status=status,
                next_stage_status=config['current']
            )
        except Exception:
            logger.exception(f"Failed to update current stage for {record['current_task_executions_id']}")
        
        # Update previous stage
        try:
            metadata_db.execute_stored_procedure(
                #5.5:Previous stage update
                'logging.usp_update_processed_flag', 
                False, 
                task_executions_id=record['previous_task_executions_id'], 
                next_stage_status=config['previous']
            )
        except Exception:
            logger.exception(f"Failed to update previous stage for {record['previous_task_executions_id']}")

##### Main execution

In [ ]:
# Load variables from input parameters 
# -------------------------------
# Handle input settings (ensure valid JSON strings)
# -------------------------------
# If source_settings is None or empty, default to empty JSON object '{}'

source_settings = source_settings or '{}'
target_settings = target_settings or '{}'

# Convert JSON strings into Python dictionaries
source_settings = json.loads(source_settings)

# -------------------------------
# Extract source configuration
# -------------------------------
# Get source schema and table names from source settings

source_schema_name = source_settings.get('schema_name')
source_table_name = source_settings.get('table_name')


# -------------------------------
# Extract target configuration
# -------------------------------
# Get target schema, table, load strategy, and primary keys

target_settings = json.loads(target_settings)
target_schema_name = target_settings.get('schema_name')
target_table_name = target_settings.get('table_name')
target_load_strategy = target_settings.get('load_strategy')
target_primary_keys = target_settings.get('primary_keys')


# -------------------------------
# Define lakehouse names (hardcoded for Bronze and Silver layers)
# -------------------------------

source_lakehouse_name = 'lh_canada_global_mds'
target_lakehouse_name = 'lh_canada_global_mds'


# -------------------------------
# Process lineage IDs from log_run_pairs
# -------------------------------
# Convert string representation of list into actual list of dictionaries
# Then extract all previous_lineage_id values into a list

lineage_ids = [record['previous_lineage_id'] for record in ast.literal_eval(log_run_pairs)]


# -------------------------------
# Get current run ID for tracking execution
# -------------------------------
# Try to get currentRunId first; fallback to activityId if not available

current_run_id = notebookutils.runtime.context.get('currentRunId')
run_id = current_run_id or notebookutils.runtime.context.get('activityId')

# -------------------------------
# Prepare logging data for current execution
# -------------------------------
# Convert log_run_pairs string to list again

log_run_pairs = ast.literal_eval(log_run_pairs)

# Add a new unique execution ID (UUID) to each record
log_run_data = [{**item, 'current_task_executions_id': str(uuid.uuid4())} for item in log_run_pairs]

In [ ]:

# -------------------------------
# Create a dictionary for logging execution start metadata
# -------------------------------

log_start_parameters = {
    'task_executions_id': '', # will be replaced later when executed
    'lineage_id': '', # will be replaced later when executed
    'run_id': run_id,
    'parent_run_id': parent_run_id,
    'task_id': task_id,
    'workspace_id': notebookutils.runtime.context.get('currentWorkspaceId'),
    'executing_object_id': notebookutils.runtime.context.get('currentNotebookId'),
    'executing_object_name': notebookutils.runtime.context.get('currentNotebookName'),
    'executing_object_run_id': notebookutils.runtime.context.get('activityId'),
    'executing_object_type': 'Notebook'
}

In [ ]:
# 1. metadata Database initialization, Retrieves secrets from Key Vault, Creates SQL connection
metadata_db = SQLDatabase.from_metadata_control_db(akv_uri, tenant_id_secret_name, client_id_secret_name, client_secret_name, server, metadata_database)

In [ ]:
#2. Get the latest watermark 

query = """
SELECT MAX(order_date) AS current_watermark_val
FROM dbo.Address
WHERE task_id = :task_id
"""

watermark = metadata_db.execute_query(
    query,
    params={"task_id": task_id}
)

max_watermark = watermark[0]["current_watermark_val"] if watermark else None

In [ ]:
# 2. Loop through each record prepared earlier in log_run_data
for record in log_run_data:

    # Set lineage_id for the current iteration
    # Comes from the previous step's lineage tracking
    log_start_parameters['lineage_id'] = record['previous_lineage_id']

    # Set task_execution_id for the current iteration
    # This was generated earlier using uuid for unique identification
    log_start_parameters['task_executions_id'] = record['current_task_executions_id']

    # Call a stored procedure in the metadata database
    # to log the execution details for this specific record
    metadata_db.execute_stored_procedure('logging.usp_log_task_execution', False, **log_start_parameters)

In [ ]:
# -------------------------------
# Build target table path
# -------------------------------

# Get the base storage path of the target lakehouse

target_lakehouse_path = notebookutils.lakehouse.get(target_lakehouse_name).properties['abfsPath']
# Construct full path to the target table (Delta table location)
target_table_path = f'{target_lakehouse_path}/Tables/{target_schema_name}/{target_table_name}'

print(f'Delta table path : {target_table_path}')

In [ ]:

import json

# Get config (URL + token)
jdbc_config_json = mssparkutils.notebook.run("nb_jdbc_connection", 60)
jdbc_config = json.loads(jdbc_config_json)


In [ ]:
# Use Spark JDBC with token
table_name = "Address"
filter_condition = "modified data > watermark"

full_table_name = f"{source_schema_name}.{source_table_name}"

query = f"(SELECT * FROM {full_table_name} WHERE {filter_condition}) AS src"

df = spark.read \
    .format("jdbc") \
    .option("url", jdbc_config["url"]) \
    .option("query", query) \
    .option("accessToken", jdbc_config["token"]) \
    .option("driver", jdbc_config["driver"]) \
    .load()


In [ ]:
status = "Failed"
try:
    sqlContext.read.format('delta').load(source_table_path).createOrReplaceTempView(f'vw_source_{source_table_name}')

    # Get only run_ids that need to be processed
    # Need to create hash, depending on load type overwrite or merges
    filter_condition = "', '".join(lineage_ids)
    primary_keys = ", ".join(target_primary_keys)

    if option_settings and 'enforce_schema' in option_settings:
        options = json.loads(option_settings)
        casted_select_str = ""
        column_list = sqlContext.read.format('delta').load(source_table_path).columns
        for col in column_list:
            if col not in options['enforce_schema']:
                casted_select_str += col + ","
            else:
                casted_select_str += f"CAST({col} as {options['enforce_schema'][col]}) as {col},"

        sql = f'''
        SELECT {casted_select_str} ROW_NUMBER() OVER(PARTITION BY {primary_keys} ORDER BY _ingest_time DESC) AS row_num
        FROM vw_source_{source_table_name}
        WHERE UPPER(_lineage_id) IN('{filter_condition}')
        '''
        logger.info(f'Option to enforce schema found - generating SELECT query with user-defined CAST instructions: {sql}')
    
    else:
        sql = f'''
        SELECT *, ROW_NUMBER() OVER(PARTITION BY {primary_keys} ORDER BY _ingest_time DESC) AS row_num
        FROM vw_source_{source_table_name}
        WHERE UPPER(_lineage_id) IN('{filter_condition}')
        '''
        logger.info(f'Generating catch-all SELECT * query: {sql}')


    # Clean up SOURCE table and add audit columns
    logger.info(f'Beginning execution - reading data from bronze source. Cleaning the table and adding audit columns')
    source_df = spark.sql(sql)
    source_df_deduped = source_df.filter(source_df.row_num == 1).drop('row_num')
    source_df_cleaned = clean_bronze_table(source_df_deduped)
    source_df_cleaned = source_df_cleaned.withColumn('_effective_start_datetime', F.lit(datetime.datetime(1900,1,1,00,00,00)))
    source_df_cleaned = source_df_cleaned.withColumn('_effective_end_datetime', F.lit(datetime.datetime(9999,12,31,23,59,59)))
    source_df_cleaned = source_df_cleaned.withColumn('_current_flag', F.lit(True))

    # Add Hash column for Primary Key
    primary_keys_snake_case = [to_snake_case(target_primary_key) for target_primary_key in target_primary_keys]
    source_df_cleaned = source_df_cleaned.withColumn('hashed_pk', F.sha2(F.concat_ws('||', *primary_keys_snake_case), 256))

    # Add Hash column for all non-primary key columns
    audit_columns = ['_ingest_time','_ingest_date','_source_file','_source_system','_table','_effective_start_datetime','_effective_end_datetime','_lineage_id','_current_flag', 'hashed_pk']
    non_key_remove_columns = primary_keys_snake_case + audit_columns
    non_key_columns = [column for column in source_df_cleaned.columns if column not in non_key_remove_columns]
    source_df_cleaned = source_df_cleaned.withColumn('hashed_row', F.sha2(F.concat_ws('||', *non_key_columns), 256))

    # Create dictionary used for UPDATE in MERGE
    set_column_list = [column for column in non_key_columns+audit_columns if column not in ['_ingest_date','_ingest_time','hashed_pk']] + ['hashed_row']
    set_column_query = {}
    for x in set_column_list:
        if x=='_effective_start_datetime':
            set_column_query["t._effective_start_datetime"] = f"'{datetime.datetime.now().isoformat()}'"
        else:
            set_column_query[f"t.{x}"] = f"s.{x}"
    

    #Check if Target exist, if exists read the original data if not create table and exit
    if not DeltaTable.isDeltaTable(spark, target_table_path):
        logger.info(f'Writing new table to {target_table_path}')
        source_df_cleaned.write.format('delta').mode("overwrite").save(target_table_path)

    else:
        match target_load_strategy:
            case 'merge':
                    # Join the df_new_data dataframe with the existing delta table using the join expression created above, updating all columns with the keys match and inserting new ones
                    # What about deletes
                    logger.info(f'Merging new records into {target_table_path}')
                    delta_table = DeltaTable.forPath(spark, target_table_path)
                    (
                        delta_table.alias("t").merge(
                            source_df_cleaned.alias("s"),
                            't.hashed_pk = s.hashed_pk'
                        ).whenMatchedUpdateAll()
                        .whenNotMatchedInsertAll()
                        .execute()
                    )

            case 'SCD1':
                    logger.info(f'Merging via SCD1 into {target_table_path}')
                    delta_table = DeltaTable.forPath(spark, target_table_path)
                    (
                        delta_table.alias("t").merge(
                            source_df_cleaned.alias("s"),
                            't.hashed_pk = s.hashed_pk'
                        ).whenMatchedUpdate(
                            condition = "(t.hashed_row != s.hashed_row) and (t._current_flag = True)",
                            set = set_column_query
                        )
                        .whenNotMatchedInsertAll()
                        .whenNotMatchedBySourceUpdate(
                            condition = "t._current_flag = True",
                            set = {"t._current_flag": "False",
                                   "t._effective_end_datetime": f"""'{datetime.datetime.now().isoformat()}'"""
                                   }
                        )
                        .execute()
                    )

            case 'SCD2-FULL':
                execution_datetime = datetime.datetime.now().isoformat()
                logger.info(f'Merging via SCD2 into {target_table_path}')

                # Step 1: Identify rows that have changed
                target_df = spark.read.format('delta').load(target_table_path)
                temp_df = source_df_cleaned.alias("source")\
                                .join(target_df.filter("_current_flag = true").alias("target"), ["hashed_pk"], how="left") \
                                .select("source.*", F.coalesce(F.col("target.hashed_row"), F.lit(None)).alias("target_hashed_row")) \
                                 .withColumn("_scd_status", F.when(F.coalesce("source.hashed_row", F.lit(1)) != F.coalesce("target_hashed_row", F.lit(1)), F.lit('NEWRECORD')).otherwise(F.lit('UNCHANGED')))\
                                .select(["_scd_status","source.*"])
                                

                new_rows_to_update_df = temp_df.alias("source")\
                                        .join(target_df.alias("target"), F.expr("target.hashed_pk = source.hashed_pk"), how="inner")\
                                        .where("target._current_flag=True and source.hashed_row <> target.hashed_row")\
                                        .select("source.*")

                logger.info(f'Found {new_rows_to_update_df.count()} updated records')

                # Step 2: Create staged updates dataframe
                    # is_insert is used to determine the SCD status. 
                    # UNCHANGED --> no changes to be made via SCD
                    # NEW RECORD --> includes both existing records that have changed, and also brand new records
                    # DEACTIVATE --> existing records that have changed, to set the record in target table to _current_flag = False

                staged_updates = (\
                    new_rows_to_update_df\
                    .withColumn("_scd_status", F.lit("DEACTIVATE"))\
                    .select(
                        F.col("_scd_status"),
                        *[F.col(c) for c in target_df.columns]
                    )\
                    .union(
                        temp_df
                        .select(
                            F.col("_scd_status"),
                            *[F.col(c) for c in target_df.columns]
                        )
                    )
                )
                staged_updates = staged_updates.drop('_effective_start_datetime')
                staged_updates = staged_updates.withColumn('_effective_start_datetime', F.lit(execution_datetime))
                

                # Step 3. Perform MERGE
                merge_condition = "t.hashed_pk = s.hashed_pk AND s._scd_status IN ('DEACTIVATE','UNCHANGED')"
                insert_values = {col_name: f"s.{col_name}" for col_name in target_df.columns}

                delta_table = DeltaTable.forPath(spark, target_table_path)
                (
                    delta_table.alias("t").merge(
                        staged_updates.alias("s"),
                        merge_condition
                    ).whenMatchedUpdate(
                        condition = "t._current_flag = True and s.hashed_row <> t.hashed_row and s._scd_status = 'DEACTIVATE'",
                        set = {"t._current_flag": "False",
                               "t._effective_end_datetime": f"""'{execution_datetime}'"""
                              }
                    ).whenNotMatchedInsert(
                        condition = "s._scd_status <> 'UNCHANGED'",
                        values = insert_values
                    )
                    .whenNotMatchedBySourceUpdate(
                        condition = "t._current_flag = True",
                        set = {"t._current_flag": "False", "t._effective_end_datetime": f"""'{execution_datetime}'"""}
                    )
                    .execute()
                )
                print(f"SCD2 merge completed successfully for {target_table_path}")



            case 'SCD2-INCREMENTAL':
                execution_datetime = datetime.datetime.now().isoformat()
                logger.info(f'Merging via SCD2 into {target_table_path}')

                # Step 1: Identify rows that have changed
                target_df = spark.read.format('delta').load(target_table_path)
                temp_df = source_df_cleaned.alias("source")\
                                .join(target_df.filter("_current_flag = true").alias("target"), ["hashed_pk"], how="left") \
                                .select("source.*", F.coalesce(F.col("target.hashed_row"), F.lit(None)).alias("target_hashed_row")) \
                                .filter(F.coalesce("source.hashed_row", F.lit(1)) != F.coalesce("target_hashed_row", F.lit(1))) \
                                .select(["source.*"])
                                
                new_rows_to_update_df = temp_df.alias("source")\
                                        .join(target_df.alias("target"), F.expr("target.hashed_pk = source.hashed_pk"), how="inner")\
                                        .where("target._current_flag=True and source.hashed_row <> target.hashed_row")\
                                        .select("source.*")

                logger.info(f'Found {new_rows_to_update_df.count()} updated records')

                # Step 2: Create staged updates dataframe
                staged_updates = (\
                    new_rows_to_update_df\
                    .withColumn("_scd_status", F.lit("Y"))\
                    .select(
                        F.col("_scd_status"),
                        *[F.col(c) for c in target_df.columns]
                    )\
                    .union(
                        temp_df
                        .withColumn("_scd_status", F.lit("N"))
                        .select(
                            F.col("_scd_status"),
                            *[F.col(c) for c in target_df.columns]
                        )
                    )
                )
                staged_updates = staged_updates.drop('_effective_start_datetime')
                staged_updates = staged_updates.withColumn('_effective_start_datetime', F.lit(execution_datetime))

                # Step 3. Perform MERGE
                merge_condition = "t.hashed_pk = s.hashed_pk AND s._scd_status='Y'"
                insert_values = {col_name: f"s.{col_name}" for col_name in target_df.columns}

                delta_table = DeltaTable.forPath(spark, target_table_path)
                (
                    delta_table.alias("t").merge(
                        staged_updates.alias("s"),
                        merge_condition
                    ).whenMatchedUpdate(
                        condition = "t._current_flag = True",
                        set = {"t._current_flag": "False",
                               "t._effective_end_datetime": f"""'{execution_datetime}'"""
                              }
                    ).whenNotMatchedInsert(
                        values = insert_values
                    )
                    .execute()
                )
                print(f"SCD2 merge completed successfully for {target_table_path}")



            # If target_delta_table_load_strategy = 'overwrite' or 'append'
            case _:
                logger.info(f'Writing new records to {target_table_path} via {target_load_strategy}')
                (
                    source_df_cleaned
                    .write.mode(target_load_strategy)
                    .format("delta")
                    .option("mergeSchema", "true")
                    .save(target_table_path)
                )

    status = "Completed"

except Exception:
    logger.exception('Processing error.')
    status = 'Failed'
    raise

finally:
    update_stage_status(status, log_run_data, metadata_db)

In [ ]:
update_stage_status(status, log_run_data, metadata_db)

In [ ]:
silver_lh_row_count = spark.read.format('delta').load(target_table_path).where(F.col("_current_flag")==True).count()

metadata_db.execute_stored_procedure(
    'logging.usp_log_update_row_count_audit',
    return_results = False,
    parent_run_id = log_start_parameters['parent_run_id'],
    stage = 'silver',
    row_count = silver_lh_row_count,
    task_id = log_start_parameters['task_id'],
    task_executions_id = log_start_parameters['task_executions_id']
)
print("Logged record counts in logging.row_count_audit")

In [ ]:
# Gather various metrics about the load/insert operation from the delta history
delta_table = DeltaTable.forPath(spark, target_table_path)
history = delta_table.history(1).select("operationMetrics")
operation_metrics = history.collect()[0]["operationMetrics"]
rows_read = source_df_cleaned.count()
rows_inserted = int(operation_metrics["numOutputRows"])
rows_updated = 0
rows_copied = rows_inserted + rows_updated

In [ ]:
# Package the metrics about the load/insert operation as a dictionary and output the result for consumption by the pipeline that invoked the notebook.
result = {
    "rows_inserted": rows_inserted,
    "rows_updated": rows_updated,
    "rows_read": rows_read,
    "rows_copied": rows_copied
}
notebookutils.notebook.exit(result)